# Training and Validation for Vapor Pressure (Chemprop CLI)

## Notebook Contract
- Uses Chemprop CLI for train/predict workflows.
- Keeps the core training function inside this notebook for transparency and reuse.
- Inputs:
  - `./Datasets/trainable_data/VP_chemprop_train_file.csv`
  - `./Datasets/trainable_data/VP_chemprop_feat_file.npz`
  - `./Datasets/trainable_data/VP_chemprop_feat_file.csv`
- Outputs per run (under `./chemprop_model_output/modelN/`):
  - trained model artifacts
  - `predictions/set_preds.csv`
  - `VP_predictions_complete.csv`
  - `run_manifest.json`


## 1. Configuration


In [ ]:
import json
import os
import re
import shutil
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from rdkit import Chem

# Run controls
RUN_TRAIN = False
RUN_PREDICT = False
RUN_FIGURES = True

# Core config
PROPERTY_CODE = 'VP'
TARGET_COLUMN = 'log10_P'
SMILES_COLUMN = 'SMILES'
SPLIT_COLUMN = 'split'

BASE_OUTPUT_DIR = Path('./chemprop_model_output')
TRAIN_DIR = Path('./Datasets/trainable_data')
TRAIN_FILE = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_train_file.csv'
PREDICTIONS_FILENAME = f'{PROPERTY_CODE}_predictions_complete.csv'

# Chemprop model/training config
BATCH_SIZE = 256
EPOCHS = 40
DEPTH = 3
DROPOUT = 0.025
RANDOM_SEED = 11

# Descriptor / transfer settings
USE_FEATURES = True
USE_TRANSFER = False
CHECKPOINT_PATH = Path('./Pretrained Model/best-epoch=19-val_loss=0.03.ckpt')
FEAT_NPZ = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_feat_file.npz'
FEAT_CSV = TRAIN_DIR / f'{PROPERTY_CODE}_chemprop_feat_file.csv'

# Evaluation selection (downstream of training)
# None -> use most recently trained model in BASE_OUTPUT_DIR
# int  -> use a specific run, e.g., 35 means ./chemprop_model_output/model35
SELECT_MODEL_NUMBER = 3

# Figure settings
# primary -> existing train/val/test figures
# low_vp  -> external low-VP liquids/solids figures
FIGURE_DATASET_MODE = 'low_vp'
RUN_LOW_VP_PREDICT = True

TOP_N_SUBGROUPS = 15
MAX_PER_ROW_SUBGROUPS = 5
PARITY_TRAIN_DENSITY_SCALE = 20.0
MAX_HEAVY_ATOMS_HIST = 40
HEAVY_ATOM_HIST_FIGSIZE = (3.5, 1.4)
PREDICTION_HEATMAP_SPLITS = ('train', 'val', 'test')
PREDICTION_HEATMAP_TEMPERATURE_BINS = 24
PREDICTION_HEATMAP_PRESSURE_BINS = 24
PREDICTION_HEATMAP_PRESSURE_RANGE = (-6.0, 3.0)
LOW_VP_TEST_DIR = Path('./Datasets/low_VP_test_sets')
LOW_VP_LIQUID_TARGET_FILE = LOW_VP_TEST_DIR / 'lowVP_liquids_test_VP_targets.csv'
LOW_VP_LIQUID_FEAT_CSV = LOW_VP_TEST_DIR / 'lowVP_liquids_test_VP_features.csv'
LOW_VP_LIQUID_FEAT_NPZ = LOW_VP_TEST_DIR / 'lowVP_liquids_test_VP_features.npz'
LOW_VP_SOLID_TARGET_FILE = LOW_VP_TEST_DIR / 'lowVP_solids_test_VP_targets.csv'
LOW_VP_SOLID_FEAT_CSV = LOW_VP_TEST_DIR / 'lowVP_solids_test_VP_features.csv'
LOW_VP_SOLID_FEAT_NPZ = LOW_VP_TEST_DIR / 'lowVP_solids_test_VP_features.npz'
LOW_VP_SPLIT_NAMES = ('low_vp_liquids', 'low_vp_solids')
LOW_VP_TEMPERATURE_TICKS = np.arange(-200, 201, 100)
LOW_VP_PARITY_PRESSURE_RANGE = (-6.0, 3.0)
LOW_VP_PARITY_PRESSURE_TICKS = np.arange(-6.0, 4.0, 2.0)
LOW_VP_HEATMAP_PRESSURE_RANGE = (-4.0, 0.0)
LOW_VP_HEATMAP_PRESSURE_TICKS = np.arange(-4.0, 1.0, 1.0)
LOW_VP_SOLID_FIGURE_EXCLUDED_SMILES = (
    'O=[N+]([O-])O.O=[N+]([O-])O.O=[N+]([O-])O.O=[N+]([O-])O.OCC(CO)(CO)CO',
)
BEST_WORST_TARGET_HIST_BINS = 24
BEST_WORST_HETEROATOM_CARBON_RATIO_BINS = 16
BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS = True
BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER = 1.5

# Error diagnostics settings
ERROR_REVIEW_SPLIT = 'test'
TOP_N_BEST_MOLECULES = 5
TOP_N_WORST_MOLECULES = 5
TOP_N_REVIEW_SUBGROUP_MOLECULES = 5
SUBGROUP_TTEST_MIN_SELECTED_COUNT = 2
SUBGROUP_TTEST_Q_THRESHOLD = 0.05
OUTLIER_Z_THRESHOLD = 3.0
OUTLIER_IQR_MULTIPLIER = 1.5
EXPORT_ERROR_REVIEW = True

print(f'Train file: {TRAIN_FILE}')
print(f'Feature NPZ: {FEAT_NPZ}')
print(f'Output root: {BASE_OUTPUT_DIR}')


## 2. Train/Predict Helpers


In [ ]:
def ensure_cli_available() -> None:
    if shutil.which('chemprop') is None:
        raise EnvironmentError("'chemprop' CLI not found in PATH. Activate your Chemprop environment first.")


def _next_model_dir(base_dir: Path) -> Path:
    base_dir.mkdir(parents=True, exist_ok=True)
    existing = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r'model\d+', d.name)
    ]
    last_num = max((int(re.search(r'\d+', d.name).group()) for d in existing), default=0)
    return base_dir / f'model{last_num + 1}'


def _latest_model_dir(base_dir: Path) -> Path | None:
    if not base_dir.exists():
        return None
    candidates = [
        d for d in base_dir.iterdir()
        if d.is_dir() and re.match(r'model\d+', d.name)
    ]
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: int(re.search(r'\d+', p.name).group()))[-1]


def _run_command(cmd: list[str], cwd: Path | None = None) -> None:
    print('Running:', ' '.join(map(str, cmd)))
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None)


def _preflight_checks(
    train_file: Path,
    smiles_col: str,
    target_col: str,
    split_col: str,
    use_features: bool,
    feat_npz: Path,
    feat_csv: Path,
    use_transfer: bool,
    checkpoint_path: Path,
) -> None:
    ensure_cli_available()

    if not train_file.exists():
        raise FileNotFoundError(f'Training file not found: {train_file}')

    train_df = pd.read_csv(train_file)
    required_cols = {smiles_col, target_col, split_col}
    missing = required_cols - set(train_df.columns)
    if missing:
        raise ValueError(f'Training file missing required columns: {sorted(missing)}')

    if use_features:
        if not feat_npz.exists():
            raise FileNotFoundError(f'Descriptor NPZ not found: {feat_npz}')
        if not feat_csv.exists():
            raise FileNotFoundError(f'Descriptor CSV not found: {feat_csv}')

    if use_transfer and not checkpoint_path.exists():
        raise FileNotFoundError(f'Transfer checkpoint not found: {checkpoint_path}')


def train_and_predict(
    property_code: str,
    target_col: str,
    smiles_col: str = 'SMILES',
    split_col: str = 'split',
    base_dir: Path = Path('chemprop_model_output'),
    train_file: Path = Path('./Datasets/trainable_data/property_chemprop_train_file.csv'),
    use_features: bool = True,
    feat_npz: Path | None = None,
    feat_csv: Path | None = None,
    use_transfer: bool = False,
    checkpoint_path: Path | None = None,
    run_train: bool = True,
    run_predict: bool = True,
    epochs: int = 100,
    batch_size: int = 8,
    depth: int = 1,
    dropout: float = 0.025,
    seed: int = 42,
) -> dict:
    feat_npz = feat_npz or Path('')
    feat_csv = feat_csv or Path('')
    checkpoint_path = checkpoint_path or Path('')

    _preflight_checks(
        train_file=train_file,
        smiles_col=smiles_col,
        target_col=target_col,
        split_col=split_col,
        use_features=use_features,
        feat_npz=feat_npz,
        feat_csv=feat_csv,
        use_transfer=use_transfer,
        checkpoint_path=checkpoint_path,
    )

    if run_train:
        model_dir = _next_model_dir(base_dir)
        model_dir.mkdir(parents=True, exist_ok=True)
    else:
        latest = _latest_model_dir(base_dir)
        if latest is None:
            raise FileNotFoundError(f'No existing model directories found in {base_dir}')
        model_dir = latest

    print(f'Using model directory: {model_dir}')

    train_cmd = [
        'chemprop', 'train',
        '--data-path', str(train_file),
        '--task-type', 'regression',
        '--splits-column', split_col,
        '--output-dir', str(model_dir),
        '--batch-size', str(batch_size),
        '--epochs', str(epochs),
        '--depth', str(depth),
        '--dropout', str(dropout),
        '--pytorch-seed', str(seed),
    ]

    if use_features:
        train_cmd.extend(['--descriptors-path', str(feat_npz)])

    if use_transfer:
        train_cmd.extend([
            '--checkpoint', str(checkpoint_path),
            '--freeze-encoder',
            '--frzn-ffn-layers', '1',
        ])

    if run_train:
        _run_command(train_cmd)

    preds_dir = model_dir / 'predictions'
    preds_dir.mkdir(parents=True, exist_ok=True)

    preds_file = preds_dir / 'set_preds.csv'
    merged_output_file = model_dir / f'{property_code}_predictions_complete.csv'

    predict_cmd = [
        'chemprop', 'predict',
        '--test-path', str(train_file),
        '--model-paths', str(model_dir),
        '--preds-path', str(preds_file),
    ]

    if use_features:
        predict_cmd.extend(['--descriptors-path', str(feat_npz)])

    if run_predict:
        _run_command(predict_cmd)

    if not preds_file.exists():
        raise FileNotFoundError(f'Predictions file not found: {preds_file}')

    preds_df = pd.read_csv(preds_file)
    train_df = pd.read_csv(train_file)

    complete_data = {
        smiles_col: train_df[smiles_col],
        f'{property_code}_true': train_df[target_col],
        f'{property_code}_pred': preds_df.iloc[:, 1],
        split_col: train_df[split_col],
    }

    if use_features:
        feat_df = pd.read_csv(feat_csv)
        if 'T_C' in feat_df.columns:
            complete_data['T_C'] = feat_df['T_C']

    complete_df = pd.DataFrame(complete_data)
    complete_df.to_csv(merged_output_file, index=False)

    manifest = {
        'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
        'property_code': property_code,
        'target_column': target_col,
        'smiles_column': smiles_col,
        'split_column': split_col,
        'train_file': str(train_file),
        'model_dir': str(model_dir),
        'preds_file': str(preds_file),
        'merged_output_file': str(merged_output_file),
        'use_features': use_features,
        'use_transfer': use_transfer,
        'checkpoint_path': str(checkpoint_path) if use_transfer else None,
        'epochs': epochs,
        'batch_size': batch_size,
        'depth': depth,
        'dropout': dropout,
        'seed': seed,
        'train_cmd': train_cmd,
        'predict_cmd': predict_cmd,
    }

    manifest_path = model_dir / 'run_manifest.json'
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)

    legacy_output_file = model_dir / PREDICTIONS_FILENAME
    if merged_output_file.name != legacy_output_file.name:
        complete_df.to_csv(legacy_output_file, index=False)

    artifacts = {
        'model_dir': model_dir,
        'preds_file': preds_file,
        'merged_output_file': merged_output_file,
        'legacy_output_file': legacy_output_file,
        'manifest_path': manifest_path,
        'metrics_csv': model_dir / 'model_0' / 'trainer_logs' / 'version_0' / 'metrics.csv',
        'best_pt': model_dir / 'model_0' / 'best.pt',
    }

    print(f'Merged predictions saved to: {merged_output_file}')
    print(f'Run manifest saved to: {manifest_path}')

    return artifacts


In [ ]:
def predict_external_vp_dataset(
    *,
    target_csv: Path,
    feat_csv: Path,
    feat_npz: Path,
    model_paths: Path,
    property_code: str,
    target_col: str,
    smiles_col: str,
    split_name: str,
    output_csv: Path,
) -> Path:
    if not target_csv.exists():
        raise FileNotFoundError(f'Target CSV not found: {target_csv}')
    if not feat_csv.exists():
        raise FileNotFoundError(f'Feature CSV not found: {feat_csv}')
    if not feat_npz.exists():
        raise FileNotFoundError(f'Feature NPZ not found: {feat_npz}')

    target_df = pd.read_csv(target_csv)
    feat_df = pd.read_csv(feat_csv)
    required_target_cols = {smiles_col, target_col}
    required_feat_cols = {smiles_col, 'T_C'}
    missing_target = required_target_cols - set(target_df.columns)
    missing_feat = required_feat_cols - set(feat_df.columns)
    if missing_target:
        raise ValueError(f'Target CSV missing required columns: {sorted(missing_target)}')
    if missing_feat:
        raise ValueError(f'Feature CSV missing required columns: {sorted(missing_feat)}')
    if len(target_df) != len(feat_df):
        raise ValueError(
            f'Target/feature row mismatch for {split_name}: {len(target_df)} vs {len(feat_df)}'
        )
    if not target_df[smiles_col].reset_index(drop=True).equals(feat_df[smiles_col].reset_index(drop=True)):
        raise ValueError(f'SMILES mismatch between target and feature CSVs for {split_name}.')

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        test_path = tmpdir_path / f'{split_name}_predict_input.csv'
        preds_path = tmpdir_path / f'{split_name}_preds.csv'

        target_df[[smiles_col]].to_csv(test_path, index=False)

        cmd = [
            'chemprop', 'predict',
            '--test-path', str(test_path),
            '--model-paths', str(model_paths),
            '--preds-path', str(preds_path),
            '--descriptors-path', str(feat_npz),
        ]
        _run_command(cmd)

        pred_df = pd.read_csv(preds_path)
        pred_cols = [c for c in pred_df.columns if c.lower() != smiles_col.lower()]
        if len(pred_cols) != 1:
            raise ValueError(f'Unexpected prediction output columns: {pred_df.columns.tolist()}')

        out_df = pd.DataFrame(
            {
                smiles_col: target_df[smiles_col].reset_index(drop=True),
                f'{property_code}_true': target_df[target_col].reset_index(drop=True),
                f'{property_code}_pred': pred_df[pred_cols[0]].reset_index(drop=True),
                SPLIT_COLUMN: split_name,
                'T_C': feat_df['T_C'].reset_index(drop=True),
            }
        )

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(output_csv, index=False)
    print(f'External predictions saved to: {output_csv}')
    return output_csv


def build_low_vp_test_predictions(
    *,
    model_dir: Path,
    property_code: str,
    target_col: str,
    smiles_col: str,
    rerun_predict: bool = False,
) -> dict:
    low_vp_dir = model_dir / 'low_vp_test_sets'
    low_vp_dir.mkdir(parents=True, exist_ok=True)

    liquids_out = low_vp_dir / f'{property_code}_low_vp_liquids_predictions.csv'
    solids_out = low_vp_dir / f'{property_code}_low_vp_solids_predictions.csv'
    combined_out = low_vp_dir / f'{property_code}_low_vp_combined_predictions.csv'

    outputs = [liquids_out, solids_out, combined_out]
    need_predict = rerun_predict or any(not path.exists() for path in outputs)
    if need_predict:
        predict_external_vp_dataset(
            target_csv=LOW_VP_LIQUID_TARGET_FILE,
            feat_csv=LOW_VP_LIQUID_FEAT_CSV,
            feat_npz=LOW_VP_LIQUID_FEAT_NPZ,
            model_paths=model_dir,
            property_code=property_code,
            target_col=target_col,
            smiles_col=smiles_col,
            split_name=LOW_VP_SPLIT_NAMES[0],
            output_csv=liquids_out,
        )
        predict_external_vp_dataset(
            target_csv=LOW_VP_SOLID_TARGET_FILE,
            feat_csv=LOW_VP_SOLID_FEAT_CSV,
            feat_npz=LOW_VP_SOLID_FEAT_NPZ,
            model_paths=model_dir,
            property_code=property_code,
            target_col=target_col,
            smiles_col=smiles_col,
            split_name=LOW_VP_SPLIT_NAMES[1],
            output_csv=solids_out,
        )
        combined_df = pd.concat([pd.read_csv(liquids_out), pd.read_csv(solids_out)], ignore_index=True)
        combined_df.to_csv(combined_out, index=False)
        print(f'Combined low-VP predictions saved to: {combined_out}')

    return {
        'output_dir': low_vp_dir,
        'liquids_predictions': liquids_out,
        'solids_predictions': solids_out,
        'combined_predictions': combined_out,
    }


## 3. Run Train/Predict


In [ ]:
run_artifacts = None
if RUN_TRAIN or RUN_PREDICT:
    run_artifacts = train_and_predict(
        property_code=PROPERTY_CODE,
        target_col=TARGET_COLUMN,
        smiles_col=SMILES_COLUMN,
        split_col=SPLIT_COLUMN,
        base_dir=BASE_OUTPUT_DIR,
        train_file=TRAIN_FILE,
        use_features=USE_FEATURES,
        feat_npz=FEAT_NPZ,
        feat_csv=FEAT_CSV,
        use_transfer=USE_TRANSFER,
        checkpoint_path=CHECKPOINT_PATH,
        run_train=RUN_TRAIN,
        run_predict=RUN_PREDICT,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        depth=DEPTH,
        dropout=DROPOUT,
        seed=RANDOM_SEED,
    )


def build_artifacts_for_model(model_dir: Path, property_code: str) -> dict:
    return {
        'model_dir': model_dir,
        'preds_file': model_dir / 'predictions' / 'set_preds.csv',
        'merged_output_file': model_dir / f'{property_code}_predictions_complete.csv',
        'legacy_output_file': model_dir / PREDICTIONS_FILENAME,
        'manifest_path': model_dir / 'run_manifest.json',
        'metrics_csv': model_dir / 'model_0' / 'trainer_logs' / 'version_0' / 'metrics.csv',
        'best_pt': model_dir / 'model_0' / 'best.pt',
    }


if SELECT_MODEL_NUMBER is None:
    if run_artifacts is not None:
        selected_model_dir = Path(run_artifacts['model_dir'])
    else:
        latest = _latest_model_dir(BASE_OUTPUT_DIR)
        if latest is None:
            raise FileNotFoundError(f'No existing model directories found in {BASE_OUTPUT_DIR}')
        selected_model_dir = latest
    selected_label = f'latest ({selected_model_dir.name})'
else:
    selected_model_dir = BASE_OUTPUT_DIR / f'model{int(SELECT_MODEL_NUMBER)}'
    if not selected_model_dir.exists():
        raise FileNotFoundError(f'Selected model directory not found: {selected_model_dir}')
    selected_label = f'model{int(SELECT_MODEL_NUMBER)}'

artifacts = build_artifacts_for_model(selected_model_dir, PROPERTY_CODE)

run_predictions_path = artifacts['merged_output_file']
if not run_predictions_path.exists() and artifacts['legacy_output_file'].exists():
    run_predictions_path = artifacts['legacy_output_file']

if not run_predictions_path.exists():
    raise FileNotFoundError(
        f'Predictions CSV not found for selected run ({selected_label}). '
        f'Looked for: {artifacts["merged_output_file"]} and {artifacts["legacy_output_file"]}'
    )

low_vp_prediction_artifacts = None
if FIGURE_DATASET_MODE == 'low_vp':
    low_vp_prediction_artifacts = build_low_vp_test_predictions(
        model_dir=selected_model_dir,
        property_code=PROPERTY_CODE,
        target_col=TARGET_COLUMN,
        smiles_col=SMILES_COLUMN,
        rerun_predict=RUN_LOW_VP_PREDICT,
    )

print(f'Downstream model selection: {selected_label}')
print('Current run artifacts used downstream:')
for k, v in artifacts.items():
    print(f'- {k}: {v}')
if low_vp_prediction_artifacts is not None:
    print('Low-VP test prediction artifacts:')
    for k, v in low_vp_prediction_artifacts.items():
        print(f'- {k}: {v}')


## 4. Evaluation Metrics


In [ ]:
metrics_df = pd.read_csv(run_predictions_path)

true_col = f'{PROPERTY_CODE}_true'
pred_col = f'{PROPERTY_CODE}_pred'

required_cols = {true_col, pred_col, SPLIT_COLUMN}
missing = required_cols - set(metrics_df.columns)
if missing:
    raise ValueError(f'Missing required columns for MAE calculation: {sorted(missing)}')

if 'summarize_target_distribution' in globals():
    target_stats_df = summarize_target_distribution(
        run_predictions_path,
        property_code=PROPERTY_CODE,
        split_col=SPLIT_COLUMN,
    )
else:
    stats_frames = []
    overall_stats_df = (
        metrics_df[true_col]
        .agg(['min', 'max', 'mean', 'median'])
        .to_frame()
        .T.assign(split='overall')
    )
    stats_frames.append(overall_stats_df)
    split_stats_df = (
        metrics_df.groupby(SPLIT_COLUMN, dropna=False)[true_col]
        .agg(['min', 'max', 'mean', 'median'])
        .reset_index()
        .rename(columns={SPLIT_COLUMN: 'split'})
    )
    stats_frames.append(split_stats_df)
    target_stats_df = pd.concat(stats_frames, ignore_index=True)[['split', 'min', 'max', 'mean', 'median']]
print('Target distribution summary:')
display(target_stats_df)

for split_name in ['train', 'val', 'test']:
    split_df = metrics_df[metrics_df[SPLIT_COLUMN] == split_name].copy()
    if split_df.empty:
        print(f'{split_name.capitalize()} MAE: split not found in {run_predictions_path}')
        continue
    mae = (split_df[true_col] - split_df[pred_col]).abs().mean()
    print(f'{split_name.capitalize()} MAE: {mae:.4f} (n={len(split_df)})')


## 5. Figure Helpers


In [ ]:
import importlib
import sys
EXTERNAL_FUNCTIONS_DIR = (Path.cwd().parent / 'External Functions').resolve()
PLOTTING_FUNCTIONS_DIR = (EXTERNAL_FUNCTIONS_DIR / 'Plotting Functions').resolve()
for path in (EXTERNAL_FUNCTIONS_DIR, PLOTTING_FUNCTIONS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import train_prediction_plotting as _train_prediction_plotting
importlib.reload(_train_prediction_plotting)

from train_prediction_plotting import (
    build_molecule_error_summary,
    build_prediction_error_table,
    compare_ranked_subgroups_to_training_for_run,
    plot_best_worst_molecules_for_run,
    plot_best_worst_heteroatom_carbon_ratio_for_run,
    plot_best_worst_target_histograms_for_run,
    plot_atom_count_hist,
    plot_heteroatom_pie,
    plot_loss_curves,
    plot_parity_scatter_for_run,
    plot_parity_scatter_grid_for_run,
    plot_prediction_heatmaps_for_run,
    plot_ranked_subgroups_for_run,
    plot_subgroups_for_run,
    plot_training_density_for_run,
    summarize_target_distribution,
)


## 6. Generate Figures


In [ ]:
if RUN_FIGURES:
    if FIGURE_DATASET_MODE == 'primary':
        figures_dir = artifacts['model_dir'] / 'figures'
        figures_dir.mkdir(parents=True, exist_ok=True)

        plot_loss_curves(artifacts['metrics_csv'], save_path=figures_dir / 'loss_curves.png')
        plot_training_density_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            density_split='train',
            density_scale=PARITY_TRAIN_DENSITY_SCALE,
            save_path=figures_dir / f'{PROPERTY_CODE}_train_density.png',
        )
        plot_parity_scatter_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            split_name='test',
            density_split='train',
            save_path=figures_dir / 'parity_plot_test.png',
        )
        plot_parity_scatter_grid_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            split_names=('train', 'val', 'test'),
            density_split='train',
            save_path=figures_dir / 'parity_plots_all_splits.png',
        )
        plot_subgroups_for_run(
            run_predictions_path,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            top_n=TOP_N_SUBGROUPS,
            max_per_row=MAX_PER_ROW_SUBGROUPS,
            outfile=None,
        )
        plot_atom_count_hist(
            run_predictions_path,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            max_atoms=MAX_HEAVY_ATOMS_HIST,
            hist_bins=39,
            figsize=HEAVY_ATOM_HIST_FIGSIZE,
            save_path=figures_dir / 'atom_count_hist.png',
        )
        plot_heteroatom_pie(
            run_predictions_path,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            save_path=figures_dir / 'heteroatom_pie.png',
        )
        plot_best_worst_molecules_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            split_name=ERROR_REVIEW_SPLIT,
            top_n=TOP_N_BEST_MOLECULES,
            bottom_n=TOP_N_WORST_MOLECULES,
            save_dir=figures_dir,
            label_mode='mae_count',
        )
        best_worst_target_summary_df = plot_best_worst_target_histograms_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            split_name=ERROR_REVIEW_SPLIT,
            top_n=TOP_N_BEST_MOLECULES,
            bottom_n=TOP_N_WORST_MOLECULES,
            target_source='true',
            bins=BEST_WORST_TARGET_HIST_BINS,
            save_path=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_target_histograms.png',
        )
        display(best_worst_target_summary_df)
        best_worst_heteroatom_carbon_ratio_df = plot_best_worst_heteroatom_carbon_ratio_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            smiles_col=SMILES_COLUMN,
            split_name=ERROR_REVIEW_SPLIT,
            top_n=TOP_N_BEST_MOLECULES,
            bottom_n=TOP_N_WORST_MOLECULES,
            bins=BEST_WORST_HETEROATOM_CARBON_RATIO_BINS,
            remove_outliers=BEST_WORST_HETEROATOM_CARBON_RATIO_REMOVE_OUTLIERS,
            outlier_iqr_multiplier=BEST_WORST_HETEROATOM_CARBON_RATIO_OUTLIER_IQR_MULTIPLIER,
            save_path=figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.png',
        )
        best_worst_heteroatom_carbon_ratio_summary_df = (
            best_worst_heteroatom_carbon_ratio_df
            .dropna(subset=['heteroatom_to_carbon_ratio'])
            .groupby('rank_group', dropna=False)
            .agg(
                n_molecules=('heteroatom_to_carbon_ratio', 'count'),
                n_plotted=('ratio_plotted_in_histogram', 'sum'),
                n_hidden_outliers=('ratio_outlier_for_histogram', 'sum'),
                min=('heteroatom_to_carbon_ratio', 'min'),
                max=('heteroatom_to_carbon_ratio', 'max'),
                mean=('heteroatom_to_carbon_ratio', 'mean'),
                median=('heteroatom_to_carbon_ratio', 'median'),
            )
            .reset_index()
        )
        display(best_worst_heteroatom_carbon_ratio_summary_df)
        best_worst_heteroatom_carbon_ratio_df.to_csv(
            figures_dir / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_best_worst_heteroatom_carbon_ratio.csv',
            index=False,
        )
        plot_prediction_heatmaps_for_run(
            run_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            split_names=PREDICTION_HEATMAP_SPLITS,
            temperature_col='T_C',
            pressure_source='true',
            temperature_bins=PREDICTION_HEATMAP_TEMPERATURE_BINS,
            pressure_bins=PREDICTION_HEATMAP_PRESSURE_BINS,
            pressure_range=PREDICTION_HEATMAP_PRESSURE_RANGE,
            error_value='absolute',
            save_path=figures_dir / f'{PROPERTY_CODE}_mean_abs_error_temperature_pressure_heatmaps.png',
        )
    elif FIGURE_DATASET_MODE == 'low_vp':
        if low_vp_prediction_artifacts is None:
            raise ValueError('Low-VP prediction artifacts are not available. Check FIGURE_DATASET_MODE and RUN_LOW_VP_PREDICT.')
        figures_dir = low_vp_prediction_artifacts['output_dir'] / 'figures'
        figures_dir.mkdir(parents=True, exist_ok=True)
        low_vp_predictions_path = low_vp_prediction_artifacts['combined_predictions']
        low_vp_figure_predictions_path = low_vp_predictions_path
        if LOW_VP_SOLID_FIGURE_EXCLUDED_SMILES:
            low_vp_fig_df = pd.read_csv(low_vp_predictions_path)
            exclusion_mask = (
                (low_vp_fig_df[SPLIT_COLUMN] == LOW_VP_SPLIT_NAMES[1])
                & low_vp_fig_df[SMILES_COLUMN].isin(set(LOW_VP_SOLID_FIGURE_EXCLUDED_SMILES))
            )
            removed_count = int(exclusion_mask.sum())
            if removed_count:
                low_vp_figure_predictions_path = figures_dir / f'{PROPERTY_CODE}_low_vp_combined_predictions_for_figures.csv'
                low_vp_fig_df.loc[~exclusion_mask].to_csv(low_vp_figure_predictions_path, index=False)
                print(
                    'Excluded '
                    f'{removed_count} low-VP solids rows from figure-only plotting input '
                    f'using {len(LOW_VP_SOLID_FIGURE_EXCLUDED_SMILES)} configured SMILES.'
                )

        plot_parity_scatter_grid_for_run(
            low_vp_figure_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            split_names=LOW_VP_SPLIT_NAMES,
            density_split=LOW_VP_SPLIT_NAMES[0],
            x_axis_range=LOW_VP_PARITY_PRESSURE_RANGE,
            y_axis_range=LOW_VP_PARITY_PRESSURE_RANGE,
            x_tick_values=LOW_VP_PARITY_PRESSURE_TICKS,
            y_tick_values=LOW_VP_PARITY_PRESSURE_TICKS,
            mae_location='bottom_right',
            save_path=figures_dir / 'parity_plots_low_vp_doublet.png',
        )
        plot_prediction_heatmaps_for_run(
            low_vp_figure_predictions_path,
            property_col=PROPERTY_CODE,
            split_col=SPLIT_COLUMN,
            split_names=LOW_VP_SPLIT_NAMES,
            temperature_col='T_C',
            pressure_source='true',
            temperature_bins=PREDICTION_HEATMAP_TEMPERATURE_BINS,
            pressure_bins=PREDICTION_HEATMAP_PRESSURE_BINS,
            pressure_range=LOW_VP_HEATMAP_PRESSURE_RANGE,
            temperature_tick_values=LOW_VP_TEMPERATURE_TICKS,
            pressure_tick_values=LOW_VP_HEATMAP_PRESSURE_TICKS,
            error_value='absolute',
            show_split_titles=False,
            show_figure_title=False,
            save_path=figures_dir / f'{PROPERTY_CODE}_low_vp_mean_abs_error_temperature_pressure_heatmaps.png',
        )
    else:
        raise ValueError(f'Unsupported FIGURE_DATASET_MODE: {FIGURE_DATASET_MODE}')

    print(f'Figures saved to: {figures_dir}')
else:
    print('RUN_FIGURES is False; skipping figure generation.')


## 7. Error Diagnostics


In [ ]:
review_df = pd.read_csv(run_predictions_path).copy()

true_col = f'{PROPERTY_CODE}_true'
pred_col = f'{PROPERTY_CODE}_pred'
required_cols = {SMILES_COLUMN, SPLIT_COLUMN, true_col, pred_col}
missing = required_cols - set(review_df.columns)
if missing:
    raise ValueError(f'Missing required columns for error diagnostics: {sorted(missing)}')

split_df = review_df[review_df[SPLIT_COLUMN] == ERROR_REVIEW_SPLIT].copy()
if split_df.empty:
    raise ValueError(f"No rows found for split='{ERROR_REVIEW_SPLIT}' in {run_predictions_path}")

split_df['residual'] = split_df[true_col] - split_df[pred_col]
split_df['abs_error'] = split_df['residual'].abs()

resid_std = split_df['residual'].std(ddof=0)
if pd.isna(resid_std) or resid_std == 0:
    split_df['residual_z'] = 0.0
else:
    split_df['residual_z'] = split_df['residual'] / resid_std

# IQR-based outlier threshold on absolute error
q1 = split_df['abs_error'].quantile(0.25)
q3 = split_df['abs_error'].quantile(0.75)
iqr = q3 - q1
iqr_threshold = q3 + OUTLIER_IQR_MULTIPLIER * iqr

split_df['flag_large_abs_error'] = split_df['abs_error'] >= iqr_threshold
split_df['flag_large_residual_z'] = split_df['residual_z'].abs() >= OUTLIER_Z_THRESHOLD
split_df['flag_potential_outlier'] = split_df['flag_large_abs_error'] | split_df['flag_large_residual_z']

error_review_df = build_prediction_error_table(
    run_predictions_path,
    property_code=PROPERTY_CODE,
    split_col=SPLIT_COLUMN,
    smiles_col=SMILES_COLUMN,
    split_name=ERROR_REVIEW_SPLIT,
    outlier_z_threshold=OUTLIER_Z_THRESHOLD,
    outlier_iqr_multiplier=OUTLIER_IQR_MULTIPLIER,
)

molecule_error_df = build_molecule_error_summary(
    run_predictions_path,
    property_code=PROPERTY_CODE,
    split_col=SPLIT_COLUMN,
    smiles_col=SMILES_COLUMN,
    split_name=ERROR_REVIEW_SPLIT,
    temperature_col='T_C',
    outlier_z_threshold=OUTLIER_Z_THRESHOLD,
    outlier_iqr_multiplier=OUTLIER_IQR_MULTIPLIER,
)

best_molecules_df = molecule_error_df.sort_values('mae', ascending=True).head(TOP_N_BEST_MOLECULES).copy()
worst_molecules_df = molecule_error_df.sort_values('mae', ascending=False).head(TOP_N_WORST_MOLECULES).copy()
outlier_df = error_review_df[error_review_df['flag_potential_outlier']].copy()

point_display_cols = [
    'error_rank', SMILES_COLUMN,
]
if 'T_C' in error_review_df.columns:
    point_display_cols.append('T_C')
point_display_cols += [
    true_col, pred_col,
    'residual', 'abs_error', 'residual_z',
    'flag_large_abs_error', 'flag_large_residual_z', 'flag_potential_outlier'
]

molecule_display_cols = [
    'best_molecule_rank', 'worst_molecule_rank', SMILES_COLUMN, 'n_entries',
]
if 'mean_temperature' in molecule_error_df.columns:
    molecule_display_cols += ['mean_temperature', 'min_temperature', 'max_temperature']
molecule_display_cols += [
    'mean_true', 'mean_pred', 'mae', 'rmse', 'mean_residual', 'max_abs_error'
]

print(f"Error diagnostics split: {ERROR_REVIEW_SPLIT} (n={len(error_review_df)} entries, {len(molecule_error_df)} molecules)")
print(f"IQR threshold (abs error): {error_review_df['abs_error_iqr_threshold'].iloc[0]:.4f}")
print(f"Potential outliers flagged: {len(outlier_df)}")

print(f"\nTop {min(TOP_N_BEST_MOLECULES, len(molecule_error_df))} best molecules by mean absolute error:")
display(best_molecules_df[molecule_display_cols])

print(f"\nTop {min(TOP_N_WORST_MOLECULES, len(molecule_error_df))} worst molecules by mean absolute error:")
display(worst_molecules_df[molecule_display_cols])

if len(outlier_df) > 0:
    print('\nFlagged potential outlier entries:')
    display(outlier_df[point_display_cols])
else:
    print('\nNo potential outliers flagged with current thresholds.')

plt.figure(figsize=(7, 4))
sns.histplot(error_review_df['abs_error'], bins=40)
plt.axvline(error_review_df['abs_error_iqr_threshold'].iloc[0], color='red', linestyle='--', label=f"IQR threshold = {error_review_df['abs_error_iqr_threshold'].iloc[0]:.3f}")
plt.xlabel('Absolute Error')
plt.ylabel('Count')
plt.title(f'Absolute Error Distribution ({ERROR_REVIEW_SPLIT} split)')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 6))
plt.scatter(error_review_df[pred_col], error_review_df['residual'], alpha=0.35, s=12)
plt.axhline(0, color='black', linestyle='--', linewidth=1)
plt.xlabel(f'Predicted {PROPERTY_CODE}')
plt.ylabel('Residual (true - pred)')
plt.title(f'Residuals vs Predictions ({ERROR_REVIEW_SPLIT} split)')
plt.tight_layout()
plt.show()

if EXPORT_ERROR_REVIEW:
    review_out_path = Path(artifacts['model_dir']) / f'{PROPERTY_CODE}_{ERROR_REVIEW_SPLIT}_error_review.csv'
    error_review_df.to_csv(review_out_path, index=False)
    print(f'\nSaved full error review table to: {review_out_path}')


In [ ]:
# Self-contained quick plot using synthetic vapor pressures
import matplotlib.pyplot as plt
import seaborn as sns

# Editable inputs
vapor_pressures_torr = [10**power for power in (-4, -2, -1, 0, 1)]
molecule_labels = [f"Molecule {i}" for i in range(1, len(vapor_pressures_torr) + 1)]

if len(molecule_labels) != len(vapor_pressures_torr):
    raise ValueError("`molecule_labels` and `vapor_pressures_torr` must have the same length.")

# Match publication/parity style conventions used in this repo
_FONT_FALLBACKS = ["Arial", "DejaVu Sans"]
_AXIS_LABEL_SIZE = 12
_TEXT_SIZE = 10
_LINE_WIDTH = 0.7
_VALIDATION_SERIES_RED = "#c65a45"

sns.set_theme(style="ticks", rc={"axes.grid": False})
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": _FONT_FALLBACKS,
        "axes.labelsize": _AXIS_LABEL_SIZE,
        "axes.labelweight": "normal",
        "xtick.labelsize": _TEXT_SIZE,
        "ytick.labelsize": _TEXT_SIZE,
        "lines.linewidth": _LINE_WIDTH,
        "axes.linewidth": _LINE_WIDTH,
        "xtick.major.width": _LINE_WIDTH,
        "ytick.major.width": _LINE_WIDTH,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
        "patch.linewidth": _LINE_WIDTH,
    }
)

fig, ax = plt.subplots(figsize=(6, 2.5), dpi=900)
point_color = _VALIDATION_SERIES_RED
x_positions = [i * 1.35 for i in range(len(molecule_labels))]
ax.scatter(x_positions, vapor_pressures_torr, color=point_color, s=55, linewidths=_LINE_WIDTH, edgecolors="white", zorder=3)
# ax.axhline(1.0, color=point_color, linestyle="--", linewidth=_LINE_WIDTH, zorder=2)
ax.set_xticks(x_positions)
ax.set_xticklabels(molecule_labels)
ax.set_xlim(min(x_positions) - 0.6, max(x_positions) + 0.6)

ax.set_yscale("log")
ax.set_ylabel("Vapor Pressure [torr]", fontsize=_AXIS_LABEL_SIZE, fontweight="normal")
ax.set_xlabel("Arbitrary Molecule", fontsize=_AXIS_LABEL_SIZE, fontweight="normal")
ax.set_title("")
ax.tick_params(axis="both", labelsize=_TEXT_SIZE, width=_LINE_WIDTH, length=6, direction="out")
ax.text(0.02, 0.96, "T = 25 °C", transform=ax.transAxes, fontsize=10, fontweight="normal", ha="left", va="top")
sns.despine(ax=ax)
fig.tight_layout()
plt.show()


## 8. Optional: Predict on Custom SMILES at a Fixed Temperature


In [ ]:
import tempfile


def predict_vp_at_temp(smiles_list: list[str], model_paths: str, temperature_celsius: float = 25.0) -> pd.DataFrame:
    if len(smiles_list) == 0:
        raise ValueError('smiles_list must contain at least one SMILES string.')

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        test_path = tmpdir_path / 'inference_smiles.csv'
        feat_npz_path = tmpdir_path / 'inference_temp.npz'
        preds_path = tmpdir_path / 'inference_preds.csv'

        pd.DataFrame({'smiles': smiles_list}).to_csv(test_path, index=False)
        temp_arr = np.full((len(smiles_list), 1), float(temperature_celsius), dtype=np.float32)
        np.savez(feat_npz_path, temp_arr)

        cmd = [
            'chemprop', 'predict',
            '--test-path', str(test_path),
            '--model-paths', str(model_paths),
            '--preds-path', str(preds_path),
            '--descriptors-path', str(feat_npz_path),
        ]
        _run_command(cmd)

        pred_df = pd.read_csv(preds_path)
        pred_cols = [c for c in pred_df.columns if c != 'smiles']
        if len(pred_cols) != 1:
            raise ValueError(f'Unexpected prediction output columns: {pred_df.columns.tolist()}')

        out_df = pd.DataFrame({
            'SMILES': smiles_list,
            'T_C': temperature_celsius,
            'VP_pred_log10': pred_df[pred_cols[0]],
        })

    return out_df


# Example usage
example_smiles = ['CCO']
checkpoint = artifacts['best_pt']
predict_vp_at_temp(example_smiles, model_paths=str(checkpoint), temperature_celsius=20.6)


## 9. Run Summary


In [ ]:
print('Run summary:')
print(f'- Property code: {PROPERTY_CODE}')
print(f'- Train file: {TRAIN_FILE}')
print(f'- Features enabled: {USE_FEATURES}')
print(f'- Model directory: {artifacts["model_dir"]}')
print(f'- Best model: {artifacts["best_pt"]}')
print(f'- Metrics CSV: {artifacts["metrics_csv"]}')
print(f'- Predictions CSV: {run_predictions_path}')
print(f'- Manifest: {artifacts["manifest_path"]}')
